# Noise & Data Quality Analysis

This notebook analyzes noise and data quality issues in the processed dataset. It identifies outliers, anomalies, and patterns that might be affecting model performance.

## Objectives
1. Understand per-product statistics and variability
2. Detect outliers using IQR method
3. Analyze zero values and their patterns
4. Examine coefficient of variation by day of week
5. Identify consecutive anomaly patterns
6. Check monthly consistency
7. Provide actionable recommendations

## Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load data
print("Loading data...")
train = pd.read_csv('../data/processed/train.csv')
test = pd.read_csv('../data/processed/test.csv')
df = pd.concat([train, test], ignore_index=True)

# Convert date column
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print("Data loaded successfully!")
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Products: {', '.join(df['item'].unique())}")
print(f"Days covered: {df['date'].nunique()}")

Loading data...
Data loaded successfully!
Dataset: 2022 rows, 32 columns
Date range: 2021-01-01 to 2025-10-30
Products: Croissant, Danish
Days covered: 1008


## 1. Per-Product Statistics

Understanding basic statistics and variability for each product. **Coefficient of Variation (CV)** is a key metric:
- CV = std/mean
- Lower CV = more predictable
- Higher CV = more noise/variability

In [3]:
for item in df['item'].unique():
    item_data = df[df['item'] == item]['quantity']
    
    print(f"\n{'='*60}")
    print(f"{item.upper()}")
    print(f"{'='*60}")
    print(f"  Mean:             {item_data.mean():8.2f}")
    print(f"  Median:           {item_data.median():8.2f}")
    print(f"  Std Dev:          {item_data.std():8.2f}")
    print(f"  CV (std/mean):    {item_data.std()/item_data.mean():8.2f}")
    print(f"  Min:              {item_data.min():8.2f}")
    print(f"  Max:              {item_data.max():8.2f}")
    print(f"  Range:            {item_data.max() - item_data.min():8.2f}")
    print(f"  25th percentile:  {item_data.quantile(0.25):8.2f}")
    print(f"  75th percentile:  {item_data.quantile(0.75):8.2f}")
    print(f"  Zero days:        {(item_data == 0).sum():8d}")
    print(f"  Skewness:         {item_data.skew():8.2f}")


CROISSANT
  Mean:                52.22
  Median:              52.00
  Std Dev:             24.98
  CV (std/mean):        0.48
  Min:                  0.00
  Max:                145.00
  Range:              145.00
  25th percentile:     35.00
  75th percentile:     66.50
  Zero days:              22
  Skewness:             0.42

DANISH
  Mean:                29.18
  Median:              29.00
  Std Dev:             15.11
  CV (std/mean):        0.52
  Min:                  0.00
  Max:                 84.00
  Range:               84.00
  25th percentile:     18.00
  75th percentile:     38.00
  Zero days:              25
  Skewness:             0.54


## 2. Outlier Detection (IQR Method)

Using the **Interquartile Range (IQR) method** to identify statistical outliers:
- Lower bound = Q1 - 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR
- Values outside these bounds are considered outliers

In [4]:
outlier_summary = []

for item in df['item'].unique():
    item_df = df[df['item'] == item].copy()
    Q1 = item_df['quantity'].quantile(0.25)
    Q3 = item_df['quantity'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = item_df[(item_df['quantity'] < lower_bound) | (item_df['quantity'] > upper_bound)]

    print(f"\n{'='*60}")
    print(f"{item.upper()}")
    print(f"{'='*60}")
    print(f"  IQR: {IQR:.2f}")
    print(f"  Lower bound: {lower_bound:.2f}")
    print(f"  Upper bound: {upper_bound:.2f}")
    print(f"  Outliers found: {len(outliers)} ({len(outliers)/len(item_df)*100:.1f}%)")

    if len(outliers) > 0:
        print(f"\n  Top 10 outlier dates:")
        outlier_details = outliers[['date', 'quantity', 'day_of_week']].sort_values('quantity', ascending=False).head(10)
        for _, row in outlier_details.iterrows():
            day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
            day_name = day_names[int(row['day_of_week'])]
            print(f"    {row['date'].date()} ({day_name}): {row['quantity']:.0f}")

        outlier_summary.append({
            'item': item,
            'outlier_count': len(outliers),
            'outlier_pct': len(outliers)/len(item_df)*100
        })


CROISSANT
  IQR: 31.50
  Lower bound: -12.25
  Upper bound: 113.75
  Outliers found: 15 (1.5%)

  Top 10 outlier dates:
    2022-06-11 (Sat): 145
    2022-11-05 (Sat): 136
    2021-08-07 (Sat): 133
    2021-08-21 (Sat): 125
    2022-07-02 (Sat): 125
    2022-05-21 (Sat): 124
    2022-07-31 (Sun): 124
    2022-06-18 (Sat): 122
    2021-12-24 (Fri): 121
    2022-07-22 (Fri): 118

DANISH
  IQR: 20.00
  Lower bound: -12.00
  Upper bound: 68.00
  Outliers found: 11 (1.1%)

  Top 10 outlier dates:
    2023-07-21 (Fri): 84
    2024-07-06 (Sat): 84
    2023-07-29 (Sat): 82
    2023-09-23 (Sat): 81
    2024-07-20 (Sat): 80
    2023-08-20 (Sun): 74
    2022-06-11 (Sat): 71
    2023-08-13 (Sun): 71
    2024-07-27 (Sat): 71
    2023-07-01 (Sat): 70


## 3. Zero Values Analysis

Zero values can indicate:
- Holiday closures (expected)
- Data collection issues (unexpected)
- Stock-outs or operational issues

High percentage of zeros can hurt model performance.

In [5]:
for item in df['item'].unique():
    item_df = df[df['item'] == item].copy()
    zeros = item_df[item_df['quantity'] == 0]

    print(f"\n{'='*60}")
    print(f"{item.upper()}")
    print(f"{'='*60}")
    print(f"  Total zero days: {len(zeros)} ({len(zeros)/len(item_df)*100:.1f}%)")

    if len(zeros) > 0:
        print(f"\n  Zero dates (first 15):")
        for date in zeros['date'].head(15):
            print(f"    {date.date()}")


CROISSANT
  Total zero days: 22 (2.2%)

  Zero dates (first 15):
    2021-01-01
    2021-01-02
    2021-01-03
    2021-01-07
    2021-01-08
    2021-01-15
    2021-01-22
    2021-01-29
    2021-07-04
    2021-11-25
    2021-12-25
    2022-01-01
    2022-11-24
    2022-12-25
    2023-01-01

DANISH
  Total zero days: 25 (2.5%)

  Zero dates (first 15):
    2021-01-01
    2021-01-02
    2021-01-03
    2021-01-07
    2021-01-08
    2021-01-15
    2021-01-22
    2021-01-29
    2021-07-04
    2021-11-25
    2021-12-25
    2022-01-01
    2022-11-24
    2022-12-25
    2023-01-01


## 4. Coefficient of Variation by Day of Week

Analyzing predictability for each day of the week:
- **Lower CV** = more consistent sales, easier to predict
- **Higher CV** = more volatile sales, harder to predict

This helps identify which days are naturally more predictable.

In [6]:
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

for item in df['item'].unique():
    print(f"\n{'='*60}")
    print(f"{item.upper()}")
    print(f"{'='*60}")
    item_df = df[df['item'] == item]

    for dow in sorted(item_df['day_of_week'].unique()):
        dow_data = item_df[item_df['day_of_week'] == dow]['quantity']
        cv = dow_data.std() / dow_data.mean() if dow_data.mean() > 0 else np.nan
        print(f"  {day_names[int(dow)]}: Mean={dow_data.mean():6.2f}, Std={dow_data.std():6.2f}, CV={cv:5.2f}")


CROISSANT
  Thu: Mean= 39.73, Std= 20.64, CV= 0.52
  Fri: Mean= 51.41, Std= 22.75, CV= 0.44
  Sat: Mean= 69.57, Std= 24.96, CV= 0.36
  Sun: Mean= 48.05, Std= 21.39, CV= 0.45

DANISH
  Thu: Mean= 23.09, Std= 12.73, CV= 0.55
  Fri: Mean= 27.42, Std= 13.87, CV= 0.51
  Sat: Mean= 37.83, Std= 15.70, CV= 0.41
  Sun: Mean= 28.33, Std= 14.09, CV= 0.50


## 5. Consecutive Anomaly Patterns

Looking for unusual consecutive patterns (large day-to-day changes) that might indicate:
- Data entry errors
- Operational disruptions
- Special events

Large jumps are defined as changes > 2 standard deviations of typical differences.

In [7]:
for item in df['item'].unique():
    item_df = df[df['item'] == item].copy().sort_values('date')
    item_df['quantity_diff'] = item_df['quantity'].diff().abs()

    # Find large jumps (> 2 std dev of differences)
    diff_threshold = item_df['quantity_diff'].std() * 2
    large_jumps = item_df[item_df['quantity_diff'] > diff_threshold]

    print(f"\n{'='*60}")
    print(f"{item.upper()}")
    print(f"{'='*60}")
    print(f"  Large jumps detected: {len(large_jumps)}")

    if len(large_jumps) > 0:
        print(f"\n  Top 10 largest day-to-day changes:")
        for idx, row in large_jumps.nlargest(10, 'quantity_diff').iterrows():
            # Get position in sorted dataframe
            pos = item_df.index.get_loc(idx)
            if pos > 0:
                prev_idx = item_df.index[pos - 1]
                prev_date = item_df.loc[prev_idx, 'date']
                prev_qty = item_df.loc[prev_idx, 'quantity']
                print(f"    {prev_date.date()} ({prev_qty:.0f}) → {row['date'].date()} ({row['quantity']:.0f}) = Δ{row['quantity_diff']:.0f}")


CROISSANT
  Large jumps detected: 186

  Top 10 largest day-to-day changes:
    2021-12-24 (121) → 2021-12-25 (0) = Δ121
    2021-07-03 (111) → 2021-07-04 (0) = Δ111
    2024-05-26 (116) → 2024-05-30 (14) = Δ102
    2021-12-31 (101) → 2022-01-01 (0) = Δ101
    2022-06-10 (54) → 2022-06-11 (145) = Δ91
    2022-12-24 (84) → 2022-12-25 (0) = Δ84
    2022-12-31 (78) → 2023-01-01 (0) = Δ78
    2023-01-07 (78) → 2023-01-08 (0) = Δ78
    2023-04-08 (93) → 2023-04-09 (15) = Δ78
    2025-03-22 (90) → 2025-03-23 (12) = Δ78

DANISH
  Large jumps detected: 199

  Top 10 largest day-to-day changes:
    2022-07-22 (8) → 2022-07-23 (62) = Δ54
    2022-08-18 (2) → 2022-08-19 (55) = Δ53
    2022-08-25 (2) → 2022-08-26 (55) = Δ53
    2022-08-14 (51) → 2022-08-18 (2) = Δ49
    2023-09-23 (81) → 2023-09-24 (33) = Δ48
    2022-09-01 (2) → 2022-09-02 (46) = Δ44
    2024-05-31 (1) → 2024-06-01 (44) = Δ43
    2024-07-06 (84) → 2024-07-07 (42) = Δ42
    2025-05-25 (58) → 2025-05-29 (16) = Δ42
    2021-12-24 (

## 6. Monthly Consistency Check

Checking if sales variability changes across months. This can reveal:
- Seasonal patterns in predictability
- Months with unusual volatility
- Changes in business operations over time

In [8]:
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

for item in df['item'].unique():
    print(f"\n{'='*60}")
    print(f"{item.upper()} - CV by Month")
    print(f"{'='*60}")
    item_df = df[df['item'] == item]

    monthly_cv = item_df.groupby('month')['quantity'].agg(['mean', 'std'])
    monthly_cv['cv'] = monthly_cv['std'] / monthly_cv['mean']

    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    for month, row in monthly_cv.iterrows():
        print(f"  {month_names[int(month)-1]}: Mean={row['mean']:6.2f}, CV={row['cv']:5.2f}")


CROISSANT - CV by Month
  Jan: Mean= 33.56, CV= 0.64
  Feb: Mean= 40.97, CV= 0.39
  Mar: Mean= 43.89, CV= 0.34
  Apr: Mean= 47.05, CV= 0.34
  May: Mean= 55.05, CV= 0.45
  Jun: Mean= 57.66, CV= 0.42
  Jul: Mean= 71.28, CV= 0.33
  Aug: Mean= 77.75, CV= 0.28
  Sep: Mean= 59.59, CV= 0.40
  Oct: Mean= 54.76, CV= 0.41
  Nov: Mean= 43.93, CV= 0.51
  Dec: Mean= 36.01, CV= 0.62

DANISH - CV by Month
  Jan: Mean= 18.43, CV= 0.57
  Feb: Mean= 25.16, CV= 0.41
  Mar: Mean= 24.10, CV= 0.35
  Apr: Mean= 26.49, CV= 0.36
  May: Mean= 31.13, CV= 0.46
  Jun: Mean= 33.49, CV= 0.44
  Jul: Mean= 42.28, CV= 0.45
  Aug: Mean= 39.41, CV= 0.47
  Sep: Mean= 32.57, CV= 0.45
  Oct: Mean= 32.40, CV= 0.33
  Nov: Mean= 23.46, CV= 0.54
  Dec: Mean= 18.04, CV= 0.58


## 7. Summary & Recommendations

### Key Metrics:
- **Overall CV**: General predictability (< 0.4 is good)
- **Zero percentage**: Impact on model training (< 5% is good)

### Warning Thresholds:
- ⚠️ CV > 0.4: High noise/variability
- ⚠️ Zeros > 5%: May hurt model performance

In [9]:
# Calculate noise metrics
for item in df['item'].unique():
    item_df = df[df['item'] == item]
    overall_cv = item_df['quantity'].std() / item_df['quantity'].mean()
    zero_pct = (item_df['quantity'] == 0).sum() / len(item_df) * 100

    print(f"\n{'='*60}")
    print(f"{item.upper()} - SUMMARY")
    print(f"{'='*60}")
    print(f"  Overall CV: {overall_cv:.2f}")
    print(f"  Zero percentage: {zero_pct:.1f}%")
    
    print("\n  Assessment:")
    if overall_cv > 0.4:
        print(f"  ⚠️  HIGH NOISE: CV > 0.4 indicates high variability")
        print(f"     → Consider using more robust models (ensemble methods)")
        print(f"     → Add more features to capture patterns")
    else:
        print(f"  ✅  GOOD: CV ≤ 0.4 indicates manageable variability")
    
    if zero_pct > 5:
        print(f"  ⚠️  MANY ZEROS: {zero_pct:.1f}% zeros might hurt model performance")
        print(f"     → Investigate zero patterns (holidays vs data issues)")
        print(f"     → Consider separate modeling for zero/non-zero days")
    else:
        print(f"  ✅  GOOD: Zero percentage is acceptable")


CROISSANT - SUMMARY
  Overall CV: 0.48
  Zero percentage: 2.2%

  Assessment:
  ⚠️  HIGH NOISE: CV > 0.4 indicates high variability
     → Consider using more robust models (ensemble methods)
     → Add more features to capture patterns
  ✅  GOOD: Zero percentage is acceptable

DANISH - SUMMARY
  Overall CV: 0.52
  Zero percentage: 2.5%

  Assessment:
  ⚠️  HIGH NOISE: CV > 0.4 indicates high variability
     → Consider using more robust models (ensemble methods)
     → Add more features to capture patterns
  ✅  GOOD: Zero percentage is acceptable


## Conclusion

This analysis provides insights into:
1. **Data Quality**: Outliers, zeros, and anomalies
2. **Predictability**: Which products and days are more predictable
3. **Noise Sources**: Where variability comes from
4. **Actionable Steps**: How to improve model performance

### Next Steps:
- Address high-impact outliers if they represent data errors
- Consider day-of-week specific models if CV varies significantly
- Investigate zero patterns to ensure they're legitimate
- Add features to capture patterns in high-CV periods